# Interactive Dashboard — Plotly

Run in order; the filter cell can be rerun with different values. For a standalone HTML dashboard, run `python dashboard.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
path=Path('data/retail_sales_synthetic.csv')
if not path.exists():path=Path('retail_sales_synthetic.csv')
df=pd.read_csv(path,parse_dates=['order_date'])

## 1. Change these filters and rerun

In [ ]:
selected_region='All'  # options: All, North, South, East, West
selected_category='All'  # options: All, Electronics, Home, Clothing, Grocery, Beauty
filtered=df.copy()
if selected_region!='All': filtered=filtered[filtered.region==selected_region]
if selected_category!='All': filtered=filtered[filtered.category==selected_category]
assert len(filtered)>0, 'No rows for chosen filters'
revenue=filtered.revenue.sum(); profit=filtered.profit.sum()
print(f'Orders: {len(filtered):,} | Revenue: ${revenue:,.2f} | Profit: ${profit:,.2f} | Margin: {100*profit/revenue:.1f}%')

## 2. Interactive category bar chart

In [ ]:
cat=filtered.groupby('category',as_index=False)['revenue'].sum().sort_values('revenue',ascending=False)
fig=px.bar(cat,x='category',y='revenue',title='Revenue by category',labels={'revenue':'Revenue (USD)'})
fig.show()

## 3. Interactive monthly trend

In [ ]:
month=filtered.set_index('order_date').resample('MS')[['revenue','profit']].sum().reset_index()
fig=px.line(month,x='order_date',y=['revenue','profit'],markers=True,title='Monthly performance')
fig.update_layout(yaxis_title='USD',legend_title_text='Metric');fig.show()

## 4. Scatter with hover detail

In [ ]:
fig=px.scatter(filtered,x='revenue',y='profit',color='region',hover_data=['order_id','category','channel'],title='Revenue vs profit by order')
fig.show()

## 5. Export HTML chart

In [ ]:
out=Path('outputs');out.mkdir(exist_ok=True)
fig.write_html(out/'interactive_scatter.html',include_plotlyjs=True)
print('Saved interactive_scatter.html')